# **PASTiSt**
## **P**oint Clouds to **A**rchitectural and **S**tructural Models of **Ti**mber **St**ructures

A comprehensive workflow for 3D beam modeling from the point clouds of timber structures.

#### Running Directory Configuration
Ensure that current working directiory is **pastist**!

In [ ]:
import os
# This code block ensures that python kernel runs at the correct root directory!!
if 'notebooks' in os.getcwd():
    %cd ..
else:
    print(os.getcwd())

## Project Manager 
The project manager facilitates creating a new project or loading an existing one.

- **Create New Project:** \
1- Select an **empty** folder to store project-related files \
2- select the point cloud to be processed in *.las, .laz, or .ply* formats. \
When a project is created, a *config.yml* file with default processing parameters is created within the project directory.

- **Load Project :** \
Select a pre-created project folder containing a *config.yml* file.

The project manager sets the "CONFIG_PATH" variable, which is used by subsequent processing steps.
 

In [ ]:
%run projectManager.py

### Config Manager 

The Config Manager allows users to view, modify, and save project configuration parameters via a simple interface in the CONFIG_PATH file (config.yml).


In [ ]:
%run configManager.py $CONFIG_PATH

## 1- Preprocessing

The preprocessing step aims to separate the cover and inner points of a structure. Initially, extracting the inner points enhances the quality of segmentation and beam detection performance in subsequent processing steps.

##### **Input parameters**
- **point_cloud**: \
Refers to the path of the input point cloud in .ply, .las, or .laz formats. It supports full path or relative path within the results directory (e.g. D:/data/point_cloud.las or results/point_cloud.ply)
- **roof_cover_voxel_size**: 0.05 \
This parameter is required to subsample the point cloud to extract an outer sparse point cloud of the roof. 
- **view_positions**: [-x, +x, -y, +y, -z, +z] \
View positions can be from 6 view points surrounding the point cloud:
x: From sides
y: From back/front
z: From below/above
Default parameters ignores "-y" and "-z" as the example point cloud does not have roof tiles on these directions. This need to be organized depending on the situation of the structure!
- **inner_point_sampling_size**: 0.01 \
Final inner point cloud is sub-sampled to reduce the number of points by this voxel size.
- **cover_inner_dist_thresh**: 0.1 \
To separate inner/outer point cloud, the threshold manages the minimum distance from cover mesh to the inner point cloud. If distance is larger than the treshold, the point assumed as the part of the inner point cloud. While smaller values makes the inner point cloud noisier, larger values can cut beam parts from the inner point cloud.
##### **Outputs**
- Cover point cloud : 01_cover_pcd.ply
- Cover mesh : 01_cover_mesh.ply
- Inner point cloud : 01_inner_pcd.ply



In [ ]:
%run runPreProcessing.py $CONFIG_PATH

In [ ]:
%run runShowResults.py $CONFIG_PATH PreProcessing

## 2- Segmentation

First stage of the segmentation step applies region-growing segmentation to the inner point cloud of the structure. The goal is to detect planar or slightly deformed surfaces using normal vectors within a local neighborhood. 

The second stage splits nonplanar segments into planar segments and nonlinear segments into linear sub-segments. This improves the number of beams detected in the beam modeling step.

#### **a- Segmentation: Region Growing**
##### **Input parameters**
- **point_cloud**: 01_inner_pcd.ply\
The input point cloud for segmentation. By default, it takes the inner point cloud generated by *PreProcessing*.
- **max_angle**: 5.0\
The maximum angle between the normal vectors of neighboring points.
- **search_radius**: 0.05 \
The maximum distance between two points that can be neighbors.
- **nr_neighbors**: 30 \
The maximum number of neighbors to check the angle constraint.
- **min_seg_size**: 300 \
The minimum number of points required for a segment to be counted.
##### **Output**
- **segments_pcd** : 02_segmented_pcd.ply \
Segmented point cloud with *segmentid* attribute column.


In [ ]:
%run runSegmentation.py $CONFIG_PATH

#### **b- Split Segments**
##### **Input parameters**
- **point_cloud**: 02_segmented_pcd.ply \
Segmented point cloud with an attribute column *segmentid*. By default, the result of *Segmentation* process.
- **surface_variation**: 0.0001\
A threshold value is computed via principal component analysis (PCA) to determine whether a segment is planar. It is estimated using the following formulation: \
 $sv = l3 / (l1 + l2 + l3)$. (Smaller is more planar!)
- **max_plane_dist**: 0.025 \
This refers to the maximum sigma-0 value of the best-fitting plane for a given segment.
- **min_seg_size**: 300 \
The minimum number of points required for a segment to be counted.
- **linear_ratio**: 0.80 \
The ratio between segment are and its minimum bounding rectangle (MBR). If estimated ratio is smaller than the given threshold, the segment is considered to be split.  \
$f_{area} = A_{Seg} / A_{MBR}$
##### **Output**
- **split_segments_pcd** : 02_split_segments_pcd.ply \
A segmented point cloud enhanced by sub-linear partitions of splittable segments.

In [ ]:
%run runSplitSegments.py $CONFIG_PATH

In [ ]:
%run runShowResults.py $CONFIG_PATH Segmentation

## 3- Beam Modeling

Beam modeling generates best fitting primitive geometry of each beam (cuboid by default) using its side face points.


#### **a- Primitive Geometry Modeling**
##### **Input parameters**
- **point_cloud**: 02_split_segments_pcd.ply \
By default, it refers to the *b-Split Segments* processing result. Result of *a-Segmentation* (02_segmented_pcd.ply) is also a valid input for beam modeling.
- **min_seg_size**: 300 \
The minimum number of points required for a segment to be counted as beam side face.
- **min_beam_width**: 0.10 \
The minimum size of a cross-section of a beam within the structure.
- **max_beam_width**: 0.40 \
The maximum size of a cross-section of a beam within the structure.
- **max_long_angle**: 3. \
The maximum angle (degree) threshold to allow two longitudinal axes of side faces are parallel to each other.
- **max_norm_angle**: 15. \
The maximum angle (degree) threshold to allow two normal vectors of side faces are nearly parallel or orthogonal to each other.

##### **Output parameters**
- **beams_mesh**: 03_beams_mesh.ply \
The name of the file that exports the modeled beams as a mesh.
- **beams_pcd**: 03_beams_pcd.ply \
The name of the file that exports the modeled beams as a point cloud. The *segmentid* attribute refers to the detected beams.

In [ ]:
%run runBeamModeling.py $CONFIG_PATH

In [ ]:
%run runShowResults.py $CONFIG_PATH BeamModeling

#### **b- Export Architectural and Structural Models**
##### **Input parameters**
- **mesh**: 03_beams_mesh.ply \
By default, it refers to the *a-Primitive Geomerty Modeling* processing result.
- **formats**: [dxf, stp] \
DXF is a well-known format that can be imported by many architectural design or CAD software programs. Exported STEP files (.stp) are specifically designed for structural analysis in DLUBAL RSTAB v8.
- **joint_detection**: true \
Joints are modeled between intersecting beams, and links them as rigid-rigid connector by default.
- **max_joint_size**: 0.20 \
The maximum size of a joint to be counted.
- **cross_sections_as_inch**: true \
If this is true, then the exported cross-sections are converted from the default meters units to inch units.
- **cluster_cross_sections**: true \
If true, it clusters similar cross-sections into single size. Otherwise, all beams are saved with originally estimated cross-section sizes.
- **material**: Nadelholz C24 \
Refers to the material of the beams. By default, it is structural softwood regarding European Standard EN 338 as C24.
##### **Output parameters**
- **out_file_Name**: 03_beams \
The prefix name of the exported results.


In [ ]:
%run runBeamExporter.py $CONFIG_PATH